# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print the dataset title and description
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Print the list of record sets and, for each, list the field `@id`s.

In [ ]:
# List all record sets and their fields by their @id
print("Available record sets (@id):\n------------------------")
for record_set in metadata.record_sets:
    print(f"- Record Set name: {record_set.name} | @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @ids for reference
record_sets = [rec_set.id for rec_set in metadata.record_sets]
print("Record sets available:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    # Store DataFrame by record set @id
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f" - Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")

# Example: Show the first 5 records of the first record set
if record_sets:
    example_id = record_sets[0]
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the first record set for demonstration (replace with specific @id as desired)
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Analyzing record set {record_set_id} ...\n")

    # Identify numeric fields by their @id and type
    record_set_obj = [rs for rs in metadata.record_sets if rs.id == record_set_id][0]
    numeric_fields = [field.id for field in record_set_obj.fields if field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'] and field.id in df.columns]
    print("Detected numeric fields (using @id):", numeric_fields)

    # Select a numeric field if any
    if numeric_fields:
        numeric_field = numeric_fields[0]

        # Drop missing values for this field
        numeric_col = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = numeric_col.mean() if numeric_col.notna().any() else 0
        filtered_df = df[numeric_col > threshold].copy()
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
        print(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_numeric = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
        filtered_df[f"{numeric_field}_normalized"] = (filtered_numeric - filtered_numeric.mean()) / filtered_numeric.std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field (if present)
        cat_fields = [field.id for field in record_set_obj.fields if field.data_type in ['Text', 'String', 'schema:Text', 'schema:String'] and field.id in df.columns]
        group_field = cat_fields[0] if cat_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No record sets present in the dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Draw a histogram and boxplot for the normalized numeric field
if record_sets and numeric_fields:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    filtered_numeric = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    sns.histplot(filtered_numeric.dropna(), bins=40, ax=axs[0], color='b')
    axs[0].set_title(f'Histogram of {numeric_field}')

    sns.boxplot(x=filtered_numeric.dropna(), ax=axs[1], color='g')
    axs[1].set_title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

    # Pairplot if there is another numeric field
    if len(numeric_fields) > 1:
        pair_df = filtered_df[numeric_fields].apply(pd.to_numeric, errors='coerce')
        sns.pairplot(pair_df.dropna())
        plt.suptitle("Pairplot of Numeric Fields", y=1.03)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.